# [LAB] Langfuse로 실행 기록하고 평가하기

LLM 애플리케이션의 실행을 Langfuse에 기록하고(Trace), 답변 품질에 점수를 남겨(Score) 봅니다. 셀의 출력은 기록을 붙이든 안 붙이든 똑같이 보입니다. 그래서 이 노트북의 성공 기준은 대부분 **Langfuse 화면**에서 확인합니다. 셀을 실행한 뒤 브라우저의 Langfuse 화면을 함께 봅니다.

- **2장 체인 실행 기록하기**: 체인을 실행할 때 기록 도구(CallbackHandler)를 넘깁니다. 보기를 고를 곳 두 군데.
- **3장 그래프 실행 기록하기**: LangGraph 그래프에 기록 도구를 기본 설정으로 붙입니다. 보기를 고를 곳 한 군데.
- **4장 체인 밖 함수 기록하기**: `@observe`로 검색 같은 일반 함수도 같은 기록에 넣습니다.
- **5~7장 점수 남기기**: 규칙, LLM 채점(LLM-as-a-Judge), 사용자 피드백으로 점수를 남깁니다.
- **8장 RAG 답변을 세 기준으로 채점하기**: 정확성·관련성·완성도를 LLM이 채점합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다.

**Langfuse 화면 열기** 브라우저에서 `.env`의 `LANGFUSE_BASE_URL` 주소를 열어 로그인하고, 키를 받은 프로젝트를 엽니다.

**내 trace 찾기** Langfuse 화면 왼쪽 메뉴의 **Tracing**을 엽니다. 목록은 가장 최근 것이 맨 위에 있습니다. 여러 사람이 같은 프로젝트를 쓰면 남의 기록도 섞여 보이므로, 다음 세 가지로 내 것을 고릅니다.

1. **이름**: 목록의 이름 칸에 보이는 이름으로 검색 칸에서 거릅니다. 이 노트북의 코드는 장마다 이름이 정해져 있습니다(각 장에서 알려 드립니다).
2. **시각**: 셀을 실행한 시각과 같은 시각의 기록을 봅니다.
3. **입력**: 기록을 열어 입력(Input)에 내가 보낸 질문이 있는지 봅니다.

기록은 몇 초 늦게 도착합니다. 셀 끝의 `flush()`까지 실행한 뒤 화면을 새로고침합니다. 코드 출력의 `대시보드`는 Langfuse 화면을, `[Scores] 탭`은 왼쪽 메뉴의 **Scores**를 말합니다.

**목차**

- 1. 실습 방법
- 2. 체인 실행 기록하기
  - 2.1 기록 도구와 체인 준비
  - 2.2 질문 하나 기록하기
  - 2.3 질문 여러 개 기록하기
- 3. 그래프 실행 기록하기
  - 3.1 날씨 도구와 Agent 노드
  - 3.2 그래프 조립과 기록 도구 붙이기
  - 3.3 실행하기
- 4. 체인 밖 함수 기록하기
  - 4.1 검색·정리 함수에 이름 붙이기
  - 4.2 RAG 흐름 묶기
  - 4.3 실행하기
  - 4.4 AI에게 물어보기
- 5. 규칙으로 채점하기
  - 5.1 답변 체인
  - 5.2 예시 포함 여부 채점하기
  - 5.3 AI에게 물어보기
- 6. LLM이 채점하기
  - 6.1 LLM Judge로 채점하기
  - 6.2 AI에게 물어보기
- 7. 사용자 피드백 기록하기
  - 7.1 사용자 피드백 점수 남기기
- 8. RAG 답변을 세 기준으로 채점하기
  - 8.1 문서 세 개
  - 8.2 채점 결과 형식
  - 8.3 검색과 답변 생성 기록하기
  - 8.4 세 기준으로 채점하는 Judge
  - 8.5 테스트 케이스
  - 8.6 실행하기
- 9. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env(["LANGFUSE"])

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 바로 위 환경 점검 셀을 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

**Langfuse에 연결하기 전에: 회사 인증서 쓰기**

Langfuse(`cloud.langfuse.com`)는 인터넷에 있는 서버라, 기록을 보낼 때 HTTPS로 연결합니다. 사내망에서는 이 연결이 회사 보안 인증서를 거칩니다. 그런데 파이썬은 기본으로 자기가 가진 인증서 목록만 믿기 때문에, 회사 인증서를 모르고 연결을 거부할 수 있습니다. 이때 오류에 `SSLError`나 `CERTIFICATE_VERIFY_FAILED`가 보입니다.

`truststore`는 파이썬이 **PC(운영체제)에 등록된 인증서**를 믿게 해 줍니다. 회사 PC에는 회사 인증서가 등록되어 있으니 연결이 됩니다. 회사 밖 PC에서 실행해도 문제없습니다.

- 아래 셀은 **Langfuse를 쓰는 셀보다 먼저** 한 번 실행합니다.
- 커널을 다시 시작했으면 이 셀부터 다시 실행합니다.

In [ ]:
import truststore

# 파이썬이 PC에 등록된 인증서(회사 인증서 포함)를 믿게 합니다
truststore.inject_into_ssl()
print("🟢 PC 인증서 사용 설정 완료")

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

## 1. 실습 방법

코드 셀에서 손댈 곳은 주석으로 표시되어 있습니다.

- `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- `# ✅ CHOOSE_HERE:` 아래에 보기가 주석으로 들어 있습니다. 줄 끝의 번호(1️⃣~4️⃣)와 설명을 보고 하나를 고릅니다. 같은 번호가 붙은 줄을 모두 선택하고 `Ctrl + /`를 눌러 주석을 푼 뒤 실행합니다.

실행한 뒤에는 **성공 기준**과 비교해 봅니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. ⛔이면 푼 줄을 다시 선택하고 `Ctrl + /`로 주석 처리한 뒤 다른 보기를 고릅니다. 마지막으로 접힌 "정답 보기"에서 보기마다의 해설을 읽습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. 체인 실행 기록하기

프롬프트 → 모델 → 출력 파서로 이어진 체인을 실행하면서 Langfuse에 기록을 남깁니다. 기록 도구는 `CallbackHandler`입니다. `.env`의 Langfuse 설정을 알아서 읽으므로 만들기만 하면 되고, 체인을 실행할 때 `config`의 `callbacks` 목록에 넣어 넘기면 그 실행의 모든 단계가 기록됩니다.

`invoke`를 한 번 호출할 때마다 trace가 하나씩 생깁니다. 이 장은 질문 하나, 그다음 질문 두 개를 실행하므로 trace가 세 개 생깁니다. 이 장의 trace 이름은 `RunnableSequence`입니다(체인의 기본 이름).

고를 곳은 두 군데입니다. 보기 중에는 오류 없이 답이 나오는데 기록만 남지 않는 것도 있습니다. 답이 나왔다고 끝내지 말고 Langfuse 화면에서 trace 수를 셉니다.

**왜 배우나:** 실행 결과만 출력해서는 모델에 실제로 어떤 프롬프트가 갔는지, 어느 단계가 오래 걸렸는지 알 수 없습니다. 기록 도구를 붙이는 것이 관찰의 첫걸음입니다.

준비 코드입니다. 필요한 도구를 불러오고, **Langfuse에 연결되는지 확인**합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다. 출력에 🟢가 나오면 다음으로 넘어갑니다. ⛔가 나오면 `.env`의 Langfuse 값 세 개를 확인하고, 이 셀의 출력을 AI에게 보여 원인을 물어봅니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

# Langfuse 서버에 연결되고 키가 맞는지 확인합니다.
# 키가 틀려도 아래 실습 코드는 오류 없이 돌아가므로, 여기서 먼저 봅니다.
try:
    langfuse_ok = get_client().auth_check()
except Exception as e:
    langfuse_ok = False
    print(f"오류 종류: {type(e).__name__}")
if langfuse_ok:
    print("🟢 Langfuse 연결 확인: 키가 맞습니다.")
else:
    print("⛔ Langfuse에 연결하지 못했습니다. .env의 LANGFUSE_ 값 세 개를 확인하세요.")

### 2.1 기록 도구와 체인 준비

In [ ]:
# CallbackHandler: .env의 LANGFUSE_* 환경변수를 자동으로 읽어 인증
# 이 핸들러를 chain.invoke()에 전달하면 모든 단계가 Langfuse에 자동 기록됨
langfuse_handler = CallbackHandler()

llm = get_llm()
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "한국어로 간결하게 답변하세요."),
        ("human", "{question}"),
    ]
)
chain = prompt | llm | StrOutputParser()

### 2.2 질문 하나 기록하기

**✅ CHOOSE_HERE: chain.invoke를 호출하면서, config의 callback으로 `langfuse_handler`를 전달하세요.**

질문 하나를 체인에 넣어 실행하고, 답변을 `result`에 담습니다. 이때 위에서 만든 `langfuse_handler`를 함께 넘겨야 이 실행이 Langfuse에 기록됩니다. 질문은 `LangGraph와 LangChain의 차이를 한 줄로 설명해줘`로 하고, 위에서 만든 프롬프트의 `{question}` 자리에 넣습니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 셀 출력에 LangGraph와 LangChain의 차이를 설명한 답변이 나온다
- 🟢 Langfuse Tracing 목록에 `RunnableSequence` trace가 새로 하나 생겼다(아래 셀까지 실행한 뒤 확인해도 됩니다)
- ⛔ 답변은 나오는데 새 trace가 없다 → 핸들러를 `config`의 `callbacks`로 넘겼는지 다시 봅니다

In [ ]:
print("=== 단일 질문 트레이싱 ===")
# config={"callbacks": [...]} — LangChain이 실행 중 각 단계마다 핸들러를 호출함
# ✅ CHOOSE_HERE: chain.invoke를 호출하면서, config의 callback으로 `langfuse_handler`를 전달하세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# result = chain.invoke(                                      # 1️⃣ 질문만 넘김
#     {"question": "LangGraph와 LangChain의 차이를 한 줄로 설명해줘"},  # 1️⃣
# )                                                           # 1️⃣
#
# result = chain.invoke(                                      # 2️⃣ config의 callbacks 목록으로 넘김
#     {"question": "LangGraph와 LangChain의 차이를 한 줄로 설명해줘"},  # 2️⃣
#     config={"callbacks": [langfuse_handler]},               # 2️⃣
# )                                                           # 2️⃣
#
# result = chain.invoke(                                      # 3️⃣ callbacks를 invoke에 바로 넘김
#     {"question": "LangGraph와 LangChain의 차이를 한 줄로 설명해줘"},  # 3️⃣
#     callbacks=[langfuse_handler],                           # 3️⃣
# )                                                           # 3️⃣
#
# result = chain.invoke(                                      # 4️⃣ callbacks에 핸들러 하나를 그대로 넣음
#     {"question": "LangGraph와 LangChain의 차이를 한 줄로 설명해줘"},  # 4️⃣
#     config={"callbacks": langfuse_handler},                 # 4️⃣
# )                                                           # 4️⃣
print(result)
print()

<details><summary>정답 보기</summary>

**정답: 2️⃣**

- 1️⃣ 핸들러를 넘기지 않았습니다. 오류 없이 답은 나오지만 기록이 남지 않습니다. 2장을 다 실행해도 `RunnableSequence` trace가 세 개가 아니라 두 개입니다.
- 2️⃣ `config`의 `callbacks` 목록에 핸들러를 넣어 넘깁니다. 체인의 세 단계(`ChatPromptTemplate` → `ChatOpenAI` → `StrOutputParser`)가 `RunnableSequence` trace 하나로 기록됩니다.
- 3️⃣ `invoke`가 `callbacks=`를 오류 없이 받지만 기록 도구로 쓰지 않습니다. 답은 나오는데 trace가 생기지 않습니다. 핸들러는 `config` 안에 넣어야 합니다.
- 4️⃣ `callbacks`에는 목록(대괄호)을 넣어야 합니다. 이 셀에서 `AttributeError: 'LangchainCallbackHandler' object has no attribute 'copy'` 오류가 납니다.

</details>

### 2.3 질문 여러 개 기록하기

**✅ CHOOSE_HERE: 반복문 안에서도 chain.invoke를 호출하면서, config의 callback으로 `langfuse_handler`를 전달하세요.**

질문 두 개를 차례로 실행합니다. 반복문 안에서 질문(`q`)마다 체인을 실행해 `answer`에 담고, 이번에도 `langfuse_handler`를 넘깁니다. 셀 끝의 `flush()`는 모아 둔 기록을 Langfuse 서버로 바로 보냅니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 셀 출력에 `Q:`와 `A:` 두 쌍이 나오고 마지막 줄에 `Tracing에서 확인`이 출력된다
- 🟢 Langfuse Tracing 목록에 이 장에서 생긴 `RunnableSequence` trace가 모두 **세 개**다
- 🟢 trace 하나를 열면 `ChatPromptTemplate` → `ChatOpenAI`(모델 호출) → `StrOutputParser` 단계가 보이고, 단계를 누르면 입력·출력과 걸린 시간이 보인다
- ⛔ 새 trace가 하나뿐이다 → 이 셀에서 고른 보기도 핸들러를 `config`의 `callbacks`로 넘기는지 봅니다
- ⛔ trace가 아직 안 보인다 → 몇 초 기다렸다가 화면을 새로고침합니다

In [ ]:
print("=== 여러 질문 배치 트레이싱 ===")
questions = [
    {"question": "RAG가 뭔가요?"},
    {"question": "Vector DB는 왜 쓰나요?"},
]
# 각 invoke마다 별도의 trace로 기록됨
for q in questions:
    # ✅ CHOOSE_HERE: 반복문 안에서도 chain.invoke를 호출하면서, config의 callback으로 `langfuse_handler`를 전달하세요.
    # (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
    # answer = chain.invoke(q, config={"callback": [langfuse_handler]})  # 1️⃣ config의 callback 키로 넘김
    # result = chain.invoke(q, config={"callbacks": [langfuse_handler]})  # 2️⃣ 답을 result에 담음
    # answer = chain.invoke(q, config={"callbacks": [langfuse_handler]}).content  # 3️⃣ 답에서 content를 꺼냄
    # answer = chain.invoke(q, config={"callbacks": [langfuse_handler]})  # 4️⃣ 질문마다 config의 callbacks로 넘김
    pass  # 그대로 둡니다
    print(f"Q: {q['question']}")
    print(f"A: {answer}")
    print()

# flush: 비동기로 큐에 쌓인 이벤트를 Langfuse 서버로 전송 완료 후 종료
# 스크립트 종료 직전에 호출하지 않으면 일부 데이터가 누락될 수 있음
langfuse = get_client()
langfuse.flush()
print("Tracing에서 확인")

<details><summary>정답 보기</summary>

**정답: 4️⃣**

- 1️⃣ 열쇠 이름이 `callbacks`가 아니라 `callback`입니다. 모르는 열쇠라 오류 없이 무시되어, 답은 나오는데 이 셀의 두 질문은 기록되지 않습니다. 2장의 trace가 하나뿐입니다.
- 2️⃣ 답을 `result`에 담아서, 바로 아래 `print(f"A: {answer}")`에서 `NameError: name 'answer' is not defined` 오류가 납니다. 첫 질문은 기록되지만 출력은 `Q:` 한 줄에서 멈춥니다.
- 3️⃣ 출력 파서(`StrOutputParser`)가 이미 답을 글자로 바꿔 돌려주므로 꺼낼 `content`가 없습니다. `AttributeError: ... has no attribute 'content'` 오류가 납니다.
- 4️⃣ 질문(`q`)마다 체인을 실행하고 앞 셀과 같은 방법으로 핸들러를 넘깁니다. `invoke` 한 번에 trace 하나씩, 이 셀에서 두 개가 생깁니다.

</details>

**볼 지점**
- 셀 출력은 기록을 붙이기 전과 똑같습니다. 기록이 남았는지는 Langfuse 화면에서만 알 수 있습니다.
- trace를 열면 체인의 세 단계가 트리로 보입니다. 모델 호출 단계에는 실제로 모델에 간 메시지(시스템 메시지 "한국어로 간결하게 답변하세요."와 질문)가 들어 있습니다.
- 모델 호출 단계에서 걸린 시간과 토큰 수도 확인합니다. 비용이 어디서 생기는지 보는 자리입니다.

## 3. 그래프 실행 기록하기

날씨 도구를 쓰는 LangGraph Agent를 기록합니다. 그래프는 agent 노드(모델)와 tools 노드(도구 실행)를 오가며 답을 만듭니다.

앞 장에서는 `invoke`를 호출할 때마다 핸들러를 넘겼습니다. 이번에는 그래프를 만들 때 `with_config`로 핸들러를 **기본 설정**으로 붙입니다. 그러면 이후 `invoke`에는 아무것도 넘기지 않아도 모든 실행이 기록됩니다. 이 장의 trace 이름은 `LangGraph`입니다.

고를 곳은 한 군데입니다. 고른 셀은 아무것도 출력하지 않으니, 아래 "실행하기" 셀까지 실행하고 Langfuse 화면을 봅니다.

**왜 배우나:** Agent는 모델과 도구를 여러 번 오갑니다. 어느 노드에서 어떤 도구를 어떤 입력으로 불렀는지, 어디서 시간이 걸렸는지를 노드 순서대로 볼 수 있어야 틀린 답의 원인을 가를 수 있습니다.

준비 코드입니다. 그래프와 도구에 필요한 것을 불러옵니다. 실행만 합니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv();

### 3.1 날씨 도구와 Agent 노드

In [ ]:
@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 반환한다."""
    data = {"서울": "맑음 23°C", "부산": "흐림 19°C", "제주": "비 17°C"}
    return data.get(city, f"{city}의 날씨 정보 없음")


tools = [get_weather]
llm = get_llm().bind_tools(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

### 3.2 그래프 조립과 기록 도구 붙이기

**✅ CHOOSE_HERE: 그래프를 컴파일 하면서, with_config 메서드를 통해 callback으로 `langfuse_handler`를 전달하세요.**

노드와 엣지를 연결한 `builder`를 실행할 수 있는 그래프로 만들고(`compile`), 거기에 `with_config`로 `langfuse_handler`를 기본 설정으로 붙여 `app`에 담습니다. 이 셀은 아무것도 출력하지 않습니다. 결과는 아래 "실행하기" 셀에서 봅니다.

보기 고르는 법: 같은 번호의 줄을 모두 선택하고 `Ctrl + /`를 누르면 주석이 풀립니다.

**성공 기준**

- 🟢 이 셀이 오류 없이 끝나고, 아래 "실행하기" 셀에서 두 질문마다 `Human Message`부터 마지막 `Ai Message`까지 칸이 나오며, 마지막 답에 도시별 날씨와 기온이 들어 있다(예: 서울은 맑고 23°C)
- 🟢 Langfuse Tracing 목록에 `LangGraph` trace가 새로 **두 개** 생겼다
- 🟢 "부산이랑 제주" trace를 열면 `agent` → `tools`(`get_weather`) → `agent` 순서로 노드가 보인다
- ⛔ 답은 나오는데 새 trace가 없다 → 고른 보기가 컴파일한 그래프(`app`)에 핸들러를 붙이는지 봅니다
- ⛔ 이 셀에서 `AttributeError`나 `TypeError`가 난다 → 핸들러를 붙이는 곳이 컴파일하기 전인지 뒤인지 봅니다

In [ ]:
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

# with_config(): 그래프 인스턴스에 기본 config를 고정
# 이후 모든 invoke() / stream() 호출에 CallbackHandler가 자동 적용됨
langfuse_handler = CallbackHandler()
# ✅ CHOOSE_HERE: 그래프를 컴파일 하면서, with_config 메서드를 통해 callback으로 `langfuse_handler`를 전달하세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + / 를 누르면 주석이 풀립니다)
# app = builder.compile().with_config({"callbacks": [langfuse_handler]})  # 1️⃣ 컴파일한 그래프에 with_config로 붙임
# app = builder.compile()                                     # 2️⃣ 컴파일만 함
# app = builder.with_config({"callbacks": [langfuse_handler]}).compile()  # 3️⃣ 컴파일 전 builder에 with_config로 붙임
# app = builder.compile(callbacks=[langfuse_handler])         # 4️⃣ compile에 callbacks를 넘김

<details><summary>정답 보기</summary>

**정답: 1️⃣**

- 1️⃣ `builder`를 컴파일한 그래프에 핸들러를 기본 설정으로 붙입니다. 이후 `app.invoke`에 아무것도 넘기지 않아도 `LangGraph` trace가 생기고, 안에 `agent`·`tools_condition`·`tools`·`get_weather`가 보입니다.
- 2️⃣ 핸들러를 붙이지 않았습니다. 오류 없이 "실행하기" 셀에서 답이 나오지만 `LangGraph` trace가 생기지 않습니다.
- 3️⃣ `builder`는 아직 설계도라 `with_config`가 없습니다. 이 셀에서 `AttributeError: 'StateGraph' object has no attribute 'with_config'` 오류가 납니다.
- 4️⃣ `compile`은 핸들러를 받지 않습니다. 이 셀에서 `TypeError: StateGraph.compile() got an unexpected keyword argument 'callbacks'` 오류가 납니다.

</details>

### 3.3 실행하기

In [ ]:
print("=== LangGraph 트레이싱 ===")
inputs = [
    "서울 날씨 알려줘",
    "부산이랑 제주 날씨 비교해줘",
]
for question in inputs:
    tracing_graph_result = app.invoke({"messages": [HumanMessage(content=question)]})
    results = tracing_graph_result["messages"]
    for m in results:
        m.pretty_print()
    print()

langfuse = get_client()
langfuse.flush()
print("Tracing에서 확인")

**볼 지점**
- `LangGraph` trace 안에 `agent`, `tools_condition`, `tools`, `get_weather`가 실행 순서대로 보입니다. `tools_condition`은 도구를 부를지 끝낼지 정한 갈림길입니다.
- `get_weather` 단계를 누르면 모델이 도구에 넘긴 입력(도시 이름)과 도구가 돌려준 날씨가 보입니다. "부산이랑 제주" 질문에서는 도구가 몇 번 호출되었는지 셉니다.
- 두 번째 `agent` 단계의 모델 호출이 도구 결과를 받아 최종 답을 씁니다.
- 셀 출력에서도 같은 흐름이 보입니다. `Ai Message` 칸의 `Tool Calls:`가 도구 호출이고, `Tool Message` 칸이 `get_weather`가 돌려준 날씨입니다.

## 4. 체인 밖 함수 기록하기

RAG처럼 "문서 검색 → 문맥 정리 → 모델 호출"로 이어지는 흐름을 기록합니다. 검색과 문맥 정리는 LangChain 체인이 아닌 일반 Python 함수라서 CallbackHandler로는 보이지 않습니다. 이런 함수에는 `@observe`를 붙입니다. `name`에 적은 이름이 Langfuse 화면에 표시됩니다.

`@observe`를 붙인 함수 안에서 다른 `@observe` 함수나 체인을 호출하면, 그 실행이 부모 아래 자식으로 묶여 트리가 됩니다. 이 장의 trace 이름은 가장 바깥 함수의 이름 `rag-pipeline`입니다. 검색은 실제로 하지 않고 정해 둔 문서 두 개를 돌려줍니다.

코드는 완성되어 있습니다. 먼저 끝까지 실행하고, `# ✍️ CHANGE_HERE:` 줄을 바꿔 다시 실행합니다.

준비 코드입니다. `observe`를 불러오고 Langfuse 클라이언트, 기록 도구, 모델을 준비합니다. 실행만 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client, observe
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
langfuse_handler = CallbackHandler()
llm = get_llm()

### 4.1 검색·정리 함수에 이름 붙이기

**✍️ CHANGE_HERE: 검색 함수가 화면에 표시될 이름을 바꿔 봅니다**

먼저 그대로 "실행하기"까지 실행하고 Langfuse 화면에서 트리를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 `name` 값을 원하는 이름(예: 내 이름을 넣은 `"retrieve-hong"`)으로 바꿉니다. 따옴표는 남깁니다. **이 셀부터 아래 셀을 차례로 다시 실행**합니다.

In [ ]:
# @observe(): 이 함수의 호출이 Langfuse에 span으로 기록됨
# name 파라미터로 대시보드에 표시될 이름을 지정할 수 있음
# ✍️ CHANGE_HERE: 검색 함수가 화면에 표시될 이름을 바꿔 봅니다
@observe(name="retrieve-documents")
def retrieve(query: str) -> list[str]:
    # 실제로는 벡터스토어에서 검색하는 부분
    # 여기서는 시뮬레이션용 더미 데이터 반환
    return [
        f"[문서1] {query}에 관한 첫 번째 참고 자료입니다.",
        f"[문서2] {query}와 관련된 두 번째 참고 자료입니다.",
    ]


@observe(name="format-context")
def format_context(docs: list[str]) -> str:
    return "\n".join(docs)

### 4.2 RAG 흐름 묶기

In [ ]:
# @observe로 감싼 함수 안에서 LangChain 체인을 호출하면
# 체인 실행이 이 span의 자식으로 자동 연결됨
@observe(name="rag-pipeline")
def rag_pipeline(question: str) -> str:
    docs = retrieve(question)
    context = format_context(docs)

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "아래 컨텍스트를 바탕으로 질문에 답하세요.\n\n{context}"),
            ("human", "{question}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()

    # CallbackHandler를 여기서도 전달해야 LLM 호출이 같은 트레이스에 묶임
    return chain.invoke(
        {"context": context, "question": question},
        config={"callbacks": [langfuse_handler]},
    )

### 4.3 실행하기

In [ ]:
print("=== @observe 데코레이터 트레이싱 ===")
questions = ["LangGraph란 무엇인가요?", "RAG의 장점은?"]
for q in questions:
    answer = rag_pipeline(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

langfuse.flush()
print("✓ 대시보드에서 retrieve → format → LLM 호출 트리를 확인하세요.")
print("Tracing에서 확인")

**성공 기준**

- 🟢 `Q:`와 `A:` 두 쌍이 나오고, 마지막에 `Tracing에서 확인`이 출력된다
- 🟢 Langfuse의 새 `rag-pipeline` trace를 열면 그 아래에 `retrieve-documents`, `format-context`, 체인(`RunnableSequence`)이 자식으로 보인다
- 🟢 이름을 바꿔 다시 실행한 뒤 새로 생긴 trace에는 `retrieve-documents` 대신 바꾼 이름이 보인다
- ⛔ `rag-pipeline`만 있고 자식이 없다 → 이 장의 셀을 준비 코드부터 차례로 다시 실행합니다
- ⛔ 바꾼 이름이 안 보인다 → 이름을 바꾼 셀과 "RAG 흐름 묶기" 셀을 다시 실행했는지 봅니다

**볼 지점**
- `retrieve-documents`를 누르면 함수의 입력(질문)과 출력(문서 두 개)이 보입니다. `@observe`가 함수의 입력과 반환값을 기록합니다.
- 체인(`RunnableSequence`)은 `rag-pipeline` 아래에 붙어 있습니다. `rag_pipeline` 안에서 체인에 핸들러를 넘겼기 때문입니다.
- 검색·정리·모델 호출이 한 trace 안에 모여 있어서, 답이 이상할 때 검색 결과부터 차례로 따라가 볼 수 있습니다.

### 4.4 AI에게 물어보기

**프롬프트 힌트** "RAG 흐름 묶기" 셀을 붙여 넣고, `chain.invoke`에서 `config`의 `callbacks`를 빼면 Langfuse의 trace 트리가 어떻게 달라지는지 설명해 달라고 합니다. 코드는 고치지 말고 설명만 해 달라고 적습니다.

**성공 기준**

- 🟢 `callbacks`를 빼면 모델 호출(체인) 기록이 트리에서 빠지고 `@observe`를 붙인 함수들만 남는다는 점을 짚었다
- 🟢 `@observe`는 붙인 함수만 기록하고, 체인 안의 단계는 CallbackHandler가 기록한다는 점을 짚었다
- ⛔ 코드를 고쳐 왔다 → 고치지 말라고 다시 적습니다
- ⛔ "아무 차이 없다"고 답했다 → 셀의 `# CallbackHandler를 여기서도 전달해야` 주석을 함께 붙여 다시 묻습니다

## 5. 규칙으로 채점하기

여기부터는 기록(Trace)에 **점수(Score)**를 붙입니다. 점수는 `langfuse.create_score`로 남기고, 어느 trace에 붙일지는 trace ID로 정합니다. 핸들러의 `last_trace_id`에 방금 실행한 trace의 ID가 들어 있습니다.

이 장은 규칙 기반 평가입니다. 답변에 예시가 들어 있는지를 코드로 판단해 1.0 또는 0.0을 `has-example`이라는 이름으로 남깁니다. 답변 체인 `run_chain`은 시스템 메시지로 "반드시 예시를 포함하세요"라고 지시합니다. 5~7장의 trace 이름은 2장과 같은 `RunnableSequence`입니다. 기록을 열어 입력(Input)의 질문으로 내 것을 고릅니다.

코드는 완성되어 있습니다. 먼저 그대로 실행하고, `# ✍️ CHANGE_HERE:` 줄을 바꿔 다시 실행합니다.

**왜 배우나:** 기록만으로는 답이 좋은지 알 수 없습니다. 점수를 남겨 두면 프롬프트를 바꿨을 때 품질이 나아졌는지 숫자로 비교할 수 있습니다.

준비 코드입니다. Langfuse 클라이언트(`langfuse`)와 모델을 준비합니다. 실행만 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
llm = get_llm()

### 5.1 답변 체인

**✍️ CHANGE_HERE: 예시를 넣으라는 지시를 빼 봅니다**

먼저 그대로 두 셀을 실행해 점수를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄에서 "반드시 예시를 포함하세요."를 지웁니다. 따옴표는 남깁니다. **이 셀과 아래 셀을 다시 실행**하고 점수가 달라지는지 봅니다.

In [ ]:
def run_chain(question: str, handler: CallbackHandler) -> str:
    prompt = ChatPromptTemplate.from_messages(
        [
            # ✍️ CHANGE_HERE: 예시를 넣으라는 지시를 빼 봅니다
            ("system", "한국어로 간결하게 답변하세요. 반드시 예시를 포함하세요."),
            ("human", "{question}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"question": question}, config={"callbacks": [handler]})

### 5.2 예시 포함 여부 채점하기

In [ ]:
# ── 1. 규칙 기반 평가 ──────────────────────────────────────────────────────────
print("=== 규칙 기반 평가 ===")
handler1 = CallbackHandler()
question1 = "Python 리스트와 튜플의 차이를 설명해줘"
answer1 = run_chain(question1, handler1)
print(f"답변: {answer1}\n")

# last_trace_id: 직전 invoke에서 생성된 trace의 ID
trace_id1 = handler1.last_trace_id

# 예시 포함 여부를 규칙으로 판단해 점수 부여
has_example = any(
    keyword in answer1 for keyword in ["예:", "예를 들어", "예시", "[", "("]
)
langfuse.create_score(
    trace_id=trace_id1,
    name="has-example",  # 대시보드에 표시될 평가 항목 이름
    value=1.0 if has_example else 0.0,
    comment="예시 포함 여부 자동 검사",
)
print(
    f"예시 포함 여부: {'✓' if has_example else '✗'} (score={1.0 if has_example else 0.0})"
)
print()


langfuse.flush()

**성공 기준**

- 🟢 답변 아래에 `예시 포함 여부: ✓ (score=1.0)` 또는 `✗ (score=0.0)` 줄이 출력된다
- 🟢 Langfuse의 새 `RunnableSequence` trace를 열면 점수(Scores)에 `has-example`과 코멘트 "예시 포함 여부 자동 검사"가 보인다
- 🟢 예시 지시를 뺀 뒤 다시 실행해 점수를 비교했다(예시가 없어 보이는데 `✓`가 나오면 그 이유를 아래 "AI에게 물어보기"에서 찾습니다)
- ⛔ 점수가 화면에 없다 → 셀 끝 `flush()`까지 실행되었는지 보고 화면을 새로고침합니다

**볼 지점**
- 점수는 코드가 정합니다. 같은 답변이면 몇 번을 채점해도 같은 점수가 나옵니다(재현성).
- 규칙은 키워드 목록 중 하나라도 답변에 들어 있으면 예시가 있다고 봅니다. 키워드 목록을 읽어 보고, 예시가 아닌데도 걸릴 만한 것이 있는지 찾아봅니다.

### 5.3 AI에게 물어보기

**프롬프트 힌트** "예시 포함 여부 채점하기" 셀을 붙여 넣고, 이 규칙이 예시가 없는 답변에도 1.0을 줄 수 있는지 리뷰해 달라고 합니다. 코드는 고치지 말고, 근거가 되는 키워드를 짚어 달라고 적습니다.

**성공 기준**

- 🟢 키워드 목록의 `(`나 `[`는 괄호만 있어도 참이 되어, 예시가 없어도 1.0이 될 수 있다는 점을 짚었다
- 🟢 이런 규칙은 빠르고 재현성이 높지만 "예시다운 예시인가" 같은 품질은 판단하지 못한다는 점을 짚었다
- ⛔ 설명 대신 고친 코드를 받았다 → 리뷰만 해 달라고 다시 적습니다
- ⛔ "규칙이 정확하다"고만 답했다 → 키워드 하나하나가 어떤 글에서 참이 되는지 다시 묻습니다

## 6. LLM이 채점하기

이번에는 다른 LLM이 답변 품질을 0.0~1.0 사이 숫자로 채점합니다(LLM-as-a-Judge). 채점 결과를 `llm-judge-quality`라는 이름으로 답변 trace에 남깁니다. Judge 호출에는 핸들러를 넘기지 않아서 Judge 자체는 따로 기록되지 않습니다. 점수만 답변 trace에 붙습니다.

이번에도 먼저 원본 그대로 결과를 본 뒤 `# ✍️ CHANGE_HERE:` 줄을 바꿉니다.

준비 코드입니다. 앞 장과 같은 모델과 답변 체인 `run_chain`을 다시 준비합니다. 실행만 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
llm = get_llm()


def run_chain(question: str, handler: CallbackHandler) -> str:
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "한국어로 간결하게 답변하세요. 반드시 예시를 포함하세요."),
            ("human", "{question}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"question": question}, config={"callbacks": [handler]})

### 6.1 LLM Judge로 채점하기

**✍️ CHANGE_HERE: Judge에게 평가 기준을 하나 더 줘 봅니다**

먼저 그대로 실행해 점수를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 따옴표 안 맨 앞에 평가 기준 한 가지를 더합니다(예: "초등학생도 이해할 수 있어야 좋은 답입니다. "). 따옴표는 남깁니다. 이 셀을 다시 실행하고 점수를 비교합니다.

In [ ]:
# ── 2. LLM-as-a-Judge 평가 ────────────────────────────────────────────────────
# 별도 LLM이 응답 품질을 0~1로 채점 — 사람 평가 없이 대규모 자동 평가에 사용
print("=== LLM-as-a-Judge 평가 ===")
handler2 = CallbackHandler()
question2 = "딥러닝이란 무엇인가요?"
answer2 = run_chain(question2, handler2)
print(f"답변: {answer2}\n")

trace_id2 = handler2.last_trace_id

judge_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            # ✍️ CHANGE_HERE: Judge에게 평가 기준을 하나 더 줘 봅니다
            "아래 답변의 품질을 0.0~1.0 사이 숫자 하나만 출력하세요. 다른 텍스트는 출력하지 마세요.",
        ),
        ("human", "질문: {question}\n답변: {answer}"),
    ]
)
judge_chain = judge_prompt | llm | StrOutputParser()

# judge LLM 호출은 평가용이므로 별도 트레이스로 기록하지 않음 (callbacks 미전달)
score_str = judge_chain.invoke({"question": question2, "answer": answer2})
try:
    score_value = max(0.0, min(1.0, float(score_str.strip())))
except ValueError:
    score_value = 0.5

langfuse.create_score(
    trace_id=trace_id2,
    name="llm-judge-quality",
    value=score_value,
    comment=f"LLM 자동 채점: {score_str.strip()}",
)
print(f"LLM 채점 점수: {score_value}")
print()


langfuse.flush()

**성공 기준**

- 🟢 답변 아래에 `LLM 채점 점수: 0.x`(또는 `1.0`)가 출력된다
- 🟢 Langfuse의 새 `RunnableSequence` trace 점수(Scores)에 `llm-judge-quality`와 코멘트 "LLM 자동 채점: …"이 보인다
- 🟢 평가 기준을 더한 뒤 점수가 달라졌는지 비교했다(LLM이 채점하므로 달라지지 않을 수도 있다)
- ⛔ 점수가 정확히 `0.5`이고 코멘트에 숫자가 아닌 글이 있다 → Judge가 숫자 하나만 답하지 않았습니다. 더한 기준 문장을 다시 봅니다

**볼 지점**
- Judge에게 준 시스템 메시지가 곧 평가 기준입니다. 지금은 "품질"이라는 말뿐이라, 무엇을 기준으로 채점했는지 알기 어렵습니다.
- 코멘트에는 Judge가 돌려준 글이 그대로 남습니다. 점수가 이상하면 코멘트부터 봅니다.

### 6.2 AI에게 물어보기

**프롬프트 힌트** "LLM Judge로 채점하기" 셀을 붙여 넣고, Judge가 숫자가 아닌 글을 돌려주면 Langfuse에 어떤 점수가 기록되는지 리뷰해 달라고 합니다. 코드는 고치지 말고 근거가 되는 줄을 짚어 달라고 적습니다.

**성공 기준**

- 🟢 숫자로 바꾸지 못하면 오류 없이 `0.5`가 기록된다는 점을 짚었다
- 🟢 그래서 화면의 0.5가 "보통 품질"인지 "채점 실패"인지 점수만으로는 구분할 수 없다는 점을 짚었다
- ⛔ 고친 코드를 먼저 내놓았다 → 고치지 말고 설명만 해 달라고 적습니다
- ⛔ "오류가 난다"고 답했다 → `try`와 `except` 부분을 다시 읽어 달라고 합니다

## 7. 사용자 피드백 기록하기

실제 서비스에서는 사용자가 누른 좋아요·싫어요를 점수로 남깁니다. 이 장은 사용자가 좋아요를 눌렀다고 가정하고 `user-feedback` 점수를 1.0으로 남깁니다. 버튼이 없으므로 값을 코드에 직접 적었습니다.

원본 그대로 한 번, 값을 바꿔 한 번 실행해 두 점수를 비교합니다.

준비 코드입니다. 앞 장과 같은 모델과 답변 체인 `run_chain`을 다시 준비합니다. 실행만 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
llm = get_llm()


def run_chain(question: str, handler: CallbackHandler) -> str:
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "한국어로 간결하게 답변하세요. 반드시 예시를 포함하세요."),
            ("human", "{question}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"question": question}, config={"callbacks": [handler]})

### 7.1 사용자 피드백 점수 남기기

**✍️ CHANGE_HERE: 싫어요(0.0)를 누른 것으로 바꿔 봅니다**

먼저 그대로 실행합니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 `1.0`을 `0.0`으로 바꿔 다시 실행하고, Langfuse 화면에서 두 trace의 점수를 비교합니다.

In [ ]:
# ── 3. 수동 평가 시뮬레이션 ──
# 실제 서비스에서는 사용자 피드백(좋아요/싫어요)을 받아 score()로 기록
print("=== 수동 평가 시뮬레이션 ===")
handler3 = CallbackHandler()
answer3 = run_chain("강화학습을 쉽게 설명해줘", handler3)
trace_id3 = handler3.last_trace_id

# 사용자가 👍를 눌렀다고 가정
# ✍️ CHANGE_HERE: 싫어요(0.0)를 누른 것으로 바꿔 봅니다
user_feedback = 1.0
langfuse.create_score(
    trace_id=trace_id3,
    name="user-feedback",
    value=user_feedback,
    comment="사용자 좋아요",
)
print(f"답변: {answer3}")
print(f"사용자 평가: 👍 (score={user_feedback})")
print()

langfuse.flush()
print("✓ 대시보드 [Scores] 탭에서 평가 결과를 확인하세요.")

**성공 기준**

- 🟢 답변과 `사용자 평가: 👍 (score=…)` 줄이 나오고, 마지막 줄에 `✓ 대시보드 [Scores] 탭에서 평가 결과를 확인하세요.`가 출력된다
- 🟢 Langfuse 왼쪽 메뉴의 **Scores** 화면에 `has-example`, `llm-judge-quality`, `user-feedback` 세 이름의 점수가 모두 보인다
- 🟢 0.0으로 바꿔 실행한 뒤에는 새 `user-feedback` 점수가 0으로 보인다
- ⛔ 세 점수 중 빠진 이름이 있다 → 해당 장의 셀을 준비 코드부터 다시 실행하고 화면을 새로고침합니다

**볼 지점**
- 0.0으로 바꿔도 출력의 👍 표시는 그대로입니다. 출력 문구는 고정된 글자이고, 실제로 남은 점수는 `score=` 뒤 숫자와 Langfuse 화면으로 확인합니다. Langfuse의 코멘트도 "사용자 좋아요" 그대로이니 점수 값만 봅니다.
- 규칙, LLM Judge, 사용자 피드백 세 가지 점수가 모두 같은 방법(`create_score`)으로 trace에 붙었습니다. 점수를 누가 매기느냐만 다릅니다.

## 8. RAG 답변을 세 기준으로 채점하기

앞 장들을 합칩니다. `@observe`로 검색(`retrieve`)과 답변 생성(`generate`)을 기록하고, LLM Judge가 답변을 **정확성·관련성·완성도** 세 기준으로 따로 채점해 trace에 점수 세 개를 남깁니다. Judge는 정해진 형식(`EvalScore`)으로만 답하게 해서, 앞 장처럼 숫자가 아닌 글을 돌려주는 일을 막습니다.

검색은 흉내만 냅니다. 문서 세 개(LangGraph 설명, RAG 설명, 날씨 글) 중 `doc_key`로 하나를 고릅니다. 세 케이스는 이렇습니다.

- CASE A: 질문에 맞는 문서
- CASE B: 문서 없음(검색 실패)
- CASE C: 엉뚱한 문서(날씨 글)

이 장의 trace 이름은 4장과 같은 `rag-pipeline`입니다. 입력(Input) 칸이 비어 있으니 실행 시각과 점수 세 개가 붙어 있는지로 고릅니다.

준비 코드입니다. `observe`, 정해진 형식을 만드는 `BaseModel`, Langfuse 클라이언트와 모델을 준비합니다. 실행만 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client, observe
from langfuse.langchain import CallbackHandler
from pydantic import BaseModel, Field

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
llm = get_llm()

### 8.1 문서 세 개

In [ ]:
# ── 가상 문서 저장소 ────────────────────────────────────────────────────────────
DOCS = {
    "langgraph": (
        "LangGraph는 LangChain 위에서 동작하는 상태 기반 멀티에이전트 프레임워크다. "
        "StateGraph로 노드(함수)와 엣지(전환 조건)를 정의하며, MemorySaver로 체크포인트를 저장한다. "
        "LangChain 0.x의 AgentExecutor를 대체하며 복잡한 루프·분기 흐름을 지원한다."
    ),
    "rag": (
        "RAG(Retrieval-Augmented Generation)는 외부 문서를 검색해 LLM의 답변 근거로 활용하는 기법이다. "
        "벡터 임베딩으로 유사 문서를 찾고, 해당 내용을 프롬프트 컨텍스트에 삽입한다. "
        "환각(hallucination)을 줄이고 최신 정보를 반영할 수 있다는 장점이 있다."
    ),
    "noise": (
        "오늘 서울 날씨는 맑고 기온은 23도이다. "
        "내일은 구름이 많고 오후에 비가 올 수 있다. "
        "주말에는 기온이 낮아질 전망이다."
    ),
}

### 8.2 채점 결과 형식

In [ ]:
# ── 평가 결과를 구조화하기 위한 Pydantic 모델 ──────────────────────────────────
class EvalScore(BaseModel):
    correctness: float = Field(description="사실 정확성 0.0~1.0")
    relevance: float = Field(description="질문과의 관련성 0.0~1.0")
    completeness: float = Field(description="답변 완성도 0.0~1.0")
    reason: str = Field(description="채점 근거 한 줄")

### 8.3 검색과 답변 생성 기록하기

In [ ]:
# ── RAG 단계별 함수 ─────────────────────────────────────────────────────────────


@observe(name="retrieve")
def retrieve(query: str, doc_key: str) -> str:
    """doc_key로 문서를 가져온다. 실제 환경에서는 벡터 검색으로 대체."""
    return DOCS.get(doc_key, "")


# 인자로 받은 기록 도구(handler)까지 입력으로 기록하려다 셀이 멈추므로, 입력 기록을 끕니다
@observe(name="generate", capture_input=False)
def generate(question: str, context: str, handler: CallbackHandler) -> str:
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "아래 컨텍스트만 참고해 질문에 답하세요. 컨텍스트가 없으면 모른다고 하세요.\n\n{context}",
            ),
            ("human", "{question}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke(
        {"context": context, "question": question},
        config={"callbacks": [handler]},
    )


# 여기도 handler를 인자로 받으므로 입력 기록을 끕니다
@observe(name="rag-pipeline", capture_input=False)
def rag_pipeline(
    question: str, doc_key: str, handler: CallbackHandler
) -> tuple[str, str]:
    context = retrieve(question, doc_key)
    answer = generate(question, context, handler)
    return answer, context

### 8.4 세 기준으로 채점하는 Judge

In [ ]:
# ── LLM Judge ──────────────────────────────────────────────────────────────────


def judge(question: str, context: str, answer: str) -> EvalScore:
    # structured output: LLM이 EvalScore 형식으로만 응답하도록 강제
    judge_llm = llm.with_structured_output(EvalScore)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "질문, 참고 컨텍스트, 답변을 보고 3개 항목을 0.0~1.0으로 채점하세요.\n"
                "- correctness: 컨텍스트 기준으로 답변이 사실적으로 맞는지\n"
                "- relevance: 답변이 질문에 직접 답하는지\n"
                "- completeness: 컨텍스트의 핵심 정보를 빠짐없이 담았는지",
            ),
            ("human", "질문: {question}\n\n컨텍스트: {context}\n\n답변: {answer}"),
        ]
    )
    chain = prompt | judge_llm
    return chain.invoke({"question": question, "context": context, "answer": answer})

### 8.5 테스트 케이스

**✍️ CHANGE_HERE: CASE C에 질문에 맞는 문서를 줘 봅니다**

먼저 그대로 "실행하기"까지 실행해 세 케이스의 점수를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 `"noise"`를 `"rag"`로 바꿉니다. **이 셀과 "실행하기" 셀을 다시 실행**하고 CASE C의 점수를 비교합니다.

In [ ]:
# ── 테스트 케이스 ───────────────────────────────────────────────────────────────

test_cases = [
    {
        "name": "CASE A — 정확한 컨텍스트",
        "question": "LangGraph가 뭔가요?",
        "doc_key": "langgraph",  # 정확한 문서
    },
    {
        "name": "CASE B — 빈 컨텍스트 (검색 실패)",
        "question": "RAG의 장점은 무엇인가요?",
        "doc_key": "missing",  # 존재하지 않는 키 → 빈 컨텍스트
    },
    {
        "name": "CASE C — 오염된 컨텍스트 (엉뚱한 문서)",
        "question": "RAG가 환각을 줄이는 원리를 설명해줘",
        # ✍️ CHANGE_HERE: CASE C에 질문에 맞는 문서를 줘 봅니다
        "doc_key": "noise",  # 날씨 문서 → 완전히 엉뚱함
    },
]

### 8.6 실행하기

In [ ]:
print("=== RAG 파이프라인 다차원 평가 ===\n")

for case in test_cases:
    print(f"[{case['name']}]")
    print(f"Q: {case['question']}")

    handler = CallbackHandler()
    answer, context = rag_pipeline(case["question"], case["doc_key"], handler)
    trace_id = handler.last_trace_id

    print(f"A: {answer[:120]}{'...' if len(answer) > 120 else ''}")

    # 3개 차원 채점
    score = judge(case["question"], context, answer)
    print(
        f"채점 → 정확성: {score.correctness:.1f} | 관련성: {score.relevance:.1f} | 완성도: {score.completeness:.1f}"
    )
    print(f"근거: {score.reason}")

    # Langfuse에 차원별 점수 기록
    for name, value in [
        ("correctness", score.correctness),
        ("relevance", score.relevance),
        ("completeness", score.completeness),
    ]:
        langfuse.create_score(trace_id=trace_id, name=name, value=value)

    print()

langfuse.flush()
print("✓ 대시보드 [Scores] 탭에서 케이스별 차이를 확인하세요.")

**성공 기준**

- 🟢 CASE A·B·C마다 `채점 → 정확성: … | 관련성: … | 완성도: …`와 `근거:` 줄이 출력된다
- 🟢 CASE A는 세 점수가 높고, CASE B는 완성도가 낮다(LLM이 채점하므로 숫자는 실행마다 다르다)
- 🟢 Langfuse의 `rag-pipeline` trace마다 점수 `correctness`·`relevance`·`completeness` 세 개가 붙어 있고, 트리에 `retrieve`와 `generate`가 보인다
- 🟢 CASE C를 `"rag"`로 바꾼 뒤에는 C의 답변이 RAG 원리를 설명하고 완성도가 올라간다
- ⛔ Judge 단계에서 오류가 난다 → 오류 메시지 전체를 AI에게 보여 원인을 물어봅니다. 모델이 정해진 형식으로 답하는 기능을 지원하지 않는 경우일 수 있으니, 풀리지 않으면 이 장은 설명만 읽고 넘어갑니다
- ⛔ 바꾼 뒤에도 C의 답변이 "정보가 없다"로 나온다 → 테스트 케이스 셀을 다시 실행했는지 봅니다

**볼 지점**
- 점수를 세 기준으로 나누면 "어디가 부족한지"가 보입니다. CASE B의 "문서에 정보가 없어 답할 수 없다"는 답은 사실이라 정확성은 높고, 질문에 답하지 못해 완성도는 낮습니다.
- CASE C는 실행마다 결과가 다를 수 있습니다. 모델이 엉뚱한 문서를 무시하고 아는 내용으로 답하면 점수가 높게 나옵니다. `근거:` 줄로 Judge가 왜 그렇게 채점했는지 봅니다.
- Tracing 목록에서 이 장의 `rag-pipeline` 세 줄을 차례로 열어 `completeness`를 비교합니다. 트리의 `retrieve`를 누르면 입력에 질문이 보여 어느 케이스인지 알 수 있습니다.

## 9. 정리

실행 기록(Trace)을 남기는 방법 세 가지와 점수(Score)를 남기는 방법 네 가지를 보았습니다.

| 장 | 방법 | Langfuse 화면에서 확인한 것 |
|---|---|---|
| 2. 체인 실행 기록하기 | `invoke`에 `callbacks`로 핸들러 넘기기 | `RunnableSequence` trace 세 개, 프롬프트 → 모델 → 파서 단계 |
| 3. 그래프 실행 기록하기 | `with_config`로 그래프에 핸들러 붙이기 | `LangGraph` trace, agent → tools → agent 노드 순서 |
| 4. 체인 밖 함수 기록하기 | `@observe(name=…)` | `rag-pipeline` 아래 검색·정리·체인 트리 |
| 5. 규칙으로 채점하기 | 코드로 판단(`has-example`) | 빠르고 재현성이 높지만 규칙이 느슨할 수 있음 |
| 6. LLM이 채점하기 | LLM Judge(`llm-judge-quality`) | 평가 기준이 흐리면 점수 근거도 흐림 |
| 7. 사용자 피드백 기록하기 | 사람의 좋아요·싫어요(`user-feedback`) | Scores 화면의 세 점수 |
| 8. RAG 답변을 세 기준으로 채점하기 | 정확성·관련성·완성도 | 어느 기준이 부족한지 나눠 보기 |

내 trace는 이름, 시각, 입력으로 찾았습니다. 기록은 몇 초 늦게 도착하므로 `flush()` 뒤 화면을 새로고침합니다.